# States

In [18]:
import pandas as pd

In [19]:
df=pd.read_csv('../data/cville_weather_cleaned.csv')

In [20]:
def classify_weather(row):
    if row["WT06"] == 1:
        return "Ice"
    
    if row["WT04"] == 1:
        return "Sleet"
    
    if row["WT18"] == 1:
        return "Snow"
    
    if row["WT03"] == 1:
        return "Thunder"
    
    if row["WT05"] == 1:
        return "Hail"
    
    if row["WT16"] == 1 or row["PRCP"] > 0:
        if row["PRCP"] > 13:
            return "Heavy-Rain"
        elif row["WT14"] == 1:
            return "Drizzle"
        elif row["TMAX"] > 83:
            return "Hot-Rainy"
        elif row["TMIN"] < 54:
            return "Cold-Rainy"
        else:
            return "Mild-Rainy"
    
    if row["WT11"] == 1:
        return "High Winds"
    
    if row["WT01"] == 1:
        return "Fog"
    
    if row["TMAX"] > 83:
        return "Hot-Dry"
    elif row["TMIN"] < 54:
        return "Cold-Dry"
    else:
        return "Mild-Dry"

In [21]:
df["state"] = df.apply(classify_weather, axis=1)

In [22]:
df["next_day"] = df["state"].shift(-1)
df["next_2_days"] = df["state"].shift(-2)
df["next_week"] = df["state"].shift(-7)
df["next_2_weeks"] = df["state"].shift(-14)

In [23]:
df = df.dropna(subset=["next_day", "next_2_days", "next_week", "next_2_weeks"])

In [24]:
df.to_csv("../data/cville_weather_cleaned.csv", index=False)

# Transition Probability Matrices

### Matrix for Full Dataset

In [33]:
import seaborn as sns
import matplotlib.pyplot as plt

transition_matrix = pd.crosstab(
    df["state"], 
    df["next_day"], 
    normalize="index"
)

plt.figure(figsize=(12, 8))

sns.heatmap(
    transition_matrix,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar=True
)

plt.title("Transition Matrix (Full Dataset)")
plt.xlabel("Next State")
plt.ylabel("Current State")

plt.tight_layout()

plt.savefig(
    "../outputs/Full_transition_matrix.png",
    dpi=75,
    bbox_inches="tight"
)
plt.close()

### Matrix for Seasons

In [32]:
import pandas as pd

season_matrices = {}

for season in df["season"].unique():
    season_df = df[df["season"] == season]

    matrix = pd.crosstab(
        season_df["state"],
        season_df["next_day"],
        normalize="index"
    )

    season_matrices[season] = matrix

for season, P in season_matrices.items():
    plt.figure(figsize=(10, 8))
    
    sns.heatmap(P, cmap="Blues", annot=False, square=True)

    plt.title(f"Transition Matrix (Season: {season})")
    plt.xlabel("Next State")
    plt.ylabel("Current State")

    plt.tight_layout()
    plt.savefig(f"../outputs/{season}_transition_matrix.png",
                dpi=75, bbox_inches="tight")
    plt.close()

# DTMC Model